# Fine-tune PP-OCRv6 cho OCR Truyện tranh (Manhwa/Webtoon)
Notebook này hướng dẫn quy trình fine-tune mô hình **PP-OCRv6** để nhận diện văn bản trong truyện tranh, tối ưu cho các font chữ đặc thù và cấu trúc bong bóng thoại.

In [ ]:
!nvidia-smi
!nvcc --version

## 1. Cài đặt Môi trường & CUDA 12.6
Bước này thực hiện cài đặt bộ toolkit CUDA cần cho nền tảng Paddle và các thư viện hỗ trợ tính toán GPU để đảm bảo hiệu suất tốt nhất khi huấn luyện mô hình.

In [ ]:
import os
# Download CUDA 12.6.0
!wget https://developer.download.nvidia.com/compute/cuda/12.6.0/local_installers/cuda-repo-ubuntu2204-12-6-local_12.6.0-560.28.03-1_amd64.deb
!dpkg -i cuda-repo-ubuntu2204-12-6-local_12.6.0-560.28.03-1_amd64.deb
!cp /var/cuda-repo-ubuntu2204-12-6-local/cuda-*-keyring.gpg /usr/share/keyrings/
!apt-get update
!apt-get -y install cuda-toolkit-12-6

In [ ]:
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda-12.6'
os.environ['PATH'] = '/usr/local/cuda-12.6/bin:' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-12.6/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

# Verify the installation
!nvcc --version

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version used by PyTorch: {torch.version.cuda}")
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

## 2. Cài đặt PaddlePaddle & PaddleOCR
Cài đặt framework PaddlePaddle phiên bản hỗ trợ CUDA 12.6 và clone mã nguồn PaddleOCR để sử dụng các script huấn luyện chuẩn.

In [ ]:
!pip install -q paddlepaddle-gpu==3.1.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q lmdb rapidfuzz paddleocr visualdl paddle2onnx==2.1.0 "paddlex[ocr]==3.7.2" onnx==1.17.0

!pip install -q --force-reinstall nvidia-cudnn-cu12==9.10.2.21 nvidia-cusparselt-cu12==0.7.1 nvidia-nccl-cu12==2.29.3

!ldconfig /usr/local/cuda-12.6/lib64

import paddle
paddle.utils.run_check()
import torch
print(f"Paddle GPU: {paddle.device.is_compiled_with_cuda()}")
print(f"Torch version: {torch.__version__}")
print(f"Torch GPU: {torch.cuda.is_available()}")

## 3. Chuẩn bị Dữ liệu (Dataset)
Tải và giải nén bộ dữ liệu Manhwa đã được gán nhãn. Dữ liệu bao gồm phần Detection (xác định vùng chứa chữ) và Recognition (nhận diện nội dung chữ).

In [ ]:
!wget https://github.com/tozydev/skanlator/releases/download/x-data/dataset.tar.gz
!tar -xzf dataset.tar.gz
!rm dataset.tar.gz

In [ ]:
import os

def count_images(path):
    if not os.path.exists(path):
        return 0
    return len([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

def count_lines(file_path):
    if not os.path.exists(file_path):
        return 0
    with open(file_path, 'r') as f:
        return sum(1 for line in f)

det_all_img_path = 'dataset/det/img'
rec_train_path = 'dataset/rec/crop_img'

det_train_label = 'dataset/det/train.txt'
det_val_label = 'dataset/det/val.txt'
rec_train_label = 'dataset/rec/train.txt'
rec_val_label = 'dataset/rec/val.txt'

print(f"--- Thống kê số lượng file ảnh ---")
print(f"Tổng số ảnh trong DET (img): {count_images(det_all_img_path)}")
print(f"Số lượng ảnh đã cắt trong REC (crop_img): {count_images(rec_train_path)}")

print(f"\n--- Thống kê số lượng mẫu (dòng trong file label) ---")
print(f"DET Train: {count_lines(det_train_label)}")
print(f"DET Val  : {count_lines(det_val_label)}")
print(f"REC Train: {count_lines(rec_train_label)}")
print(f"REC Val  : {count_lines(rec_val_label)}")

In [ ]:
print("--- 5 dòng đầu của det/train.txt ---")
!head -n 5 dataset/det/train.txt

print("\n--- 5 dòng đầu của rec/train.txt ---")
!head -n 5 dataset/rec/train.txt

## 4. Tải Mô hình Pretrained
Chúng ta bắt đầu từ các trọng số đã được huấn luyện sẵn của PP-OCRv6 để tiết kiệm thời gian và tăng độ chính xác.

In [ ]:
!mkdir -p pretrained

!wget -P pretrained https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_small_det_pretrained.pdparams
!wget -P pretrained https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_small_rec_pretrained.pdparams

!ls -lh pretrained

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

## 5. Cấu hình & Huấn luyện (Fine-tuning)
Thiết lập các tham số như số lượng epoch, step và đường dẫn lưu mô hình. Sau đó chạy script `train.py` cho cả hai thành phần Det và Rec.

### 5.1. Thiết lập Cấu hình (Configuration)
Chúng ta định nghĩa các tham số quan trọng trong `config_dict` để ghi đè lên file `.yml` mặc định:
- **Global**: Lưu trữ đường dẫn mô hình pretrained, số lượng epoch (vòng lặp) và thư mục lưu kết quả.
- **Train/Eval**: Chỉ định đường dẫn tới tập dữ liệu ảnh và file nhãn (label) tương ứng.

In [ ]:
det_onnx_model_dir = "output/PP-OCRv6_small_det_onnx"
rec_onnx_model_dir = "output/PP-OCRv6_small_rec_onnx"

det_best_model_path = "output/PP-OCRv6_small_det/best_model/model"
rec_best_model_path = "output/PP-OCRv6_small_rec/best_model/model"

det_infer_model_dir = "output/PP-OCRv6_small_det_infer"
rec_infer_model_dir = "output/PP-OCRv6_small_rec_infer"

det_train_config = {
    "Global": {
        "save_model_dir": "output/PP-OCRv6_small_det/",
        "pretrained_model": "pretrained/PP-OCRv6_small_det_pretrained",
        "epoch_num": 40,
        "save_epoch_step": 10,
        "eval_batch_step": [0, 226],
        "use_visualdl": True
    },
    "Train": {
        "dataset": {
            "data_dir": "dataset/det/",
            "label_file_list": ["dataset/det/train.txt"]
        },
    },
    "Eval": {
        "dataset": {
            "data_dir": "dataset/det/",
            "label_file_list": ["dataset/det/val.txt"]
        },
    }
}

det_val_config = {
    "Global": {
        "pretrained_model": det_best_model_path,
    },
}

det_export_config = {
     "Global": {
         "pretrained_model": det_best_model_path,
         "save_inference_dir": det_infer_model_dir,
     },
 }

rec_train_config = {
    "Global": {
        "save_model_dir": "output/PP-OCRv6_small_rec/",
        "pretrained_model": "pretrained/PP-OCRv6_small_rec_pretrained",
        "epoch_num": 20,
        "save_epoch_step": 5,
        "eval_batch_step": [0, 92],
        "use_visualdl": True,
        "character_dict_path": "PaddleOCR/ppocr/utils/dict/ppocrv6_dict.txt",
    },
    "Train": {
        "dataset": {
            "data_dir": "dataset/rec/",
            "label_file_list": ["dataset/rec/train.txt"]
        },
    },
    "Eval": {
        "dataset": {
            "data_dir": "dataset/rec/",
            "label_file_list": ["dataset/rec/val.txt"]
        },
    }
}

rec_val_config = {
     "Global": {
        "pretrained_model": rec_best_model_path,
    },
}

rec_export_config = {
    "Global": {
        "pretrained_model": rec_best_model_path,
        "save_inference_dir": rec_infer_model_dir,
    },
 }


In [ ]:
import yaml
import copy

def dict_to_args(config_dict, prefix=""):
    args = []
    for k, v in config_dict.items():
        new_key = f"{prefix}.{k}" if prefix else k
        if isinstance(v, dict):
            args.extend(dict_to_args(v, new_key))
        else:
            if v is True: val_str = "True"
            elif v is False: val_str = "False"
            elif v is None: val_str = "null"
            elif isinstance(v, list):
                items = []
                for item in v:
                    items.append(str(item))
                list_content = f"[{','.join(items)}]"
                val_str = f"'{list_content}'"
            else:
                val_str = str(v)

            args.append(f"{new_key}={val_str}")
    return args

def merge_config(base_config, override_config):
    result = copy.deepcopy(base_config)
    for key, value in override_config.items():
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = merge_config(result[key], value)
        else:
            result[key] = copy.deepcopy(value)
    return result

det_train_args = " ".join(dict_to_args(det_train_config))
rec_train_args = " ".join(dict_to_args(rec_train_config))

det_val_args = " ".join(dict_to_args(merge_config(det_train_config, det_val_config)))
rec_val_args = " ".join(dict_to_args(merge_config(rec_train_config, rec_val_config)))

det_export_args = " ".join(dict_to_args(merge_config(det_train_config, det_export_config)))
rec_export_args = " ".join(dict_to_args(merge_config(rec_train_config, rec_export_config)))

### 5.2. Chạy Huấn luyện (Fine-tuning Execution)
Sử dụng script `tools/train.py` của PaddleOCR. Tham số `-c` chỉ định file cấu hình gốc của PP-OCRv6, và `-o` dùng để nạp các tùy chỉnh chúng ta vừa thiết lập ở bước trên.

In [ ]:
!python3 PaddleOCR/tools/train.py -c PaddleOCR/configs/det/PP-OCRv6/PP-OCRv6_small_det.yml -o {det_train_args}

In [ ]:
!python3 PaddleOCR/tools/train.py -c PaddleOCR/configs/rec/PP-OCRv6/PP-OCRv6_small_rec.yml -o {rec_train_args}

### 5.3. Đánh giá Mô hình (Evaluation)
Sau khi huấn luyện, chúng ta chạy script `tools/eval.py` trên tập **Validation** để kiểm tra các chỉ số như Precision, Recall và Hmean. Điều này giúp đảm bảo mô hình không bị quá khớp (overfitting).

In [ ]:
!python3 PaddleOCR/tools/eval.py -c PaddleOCR/configs/det/PP-OCRv6/PP-OCRv6_small_det.yml -o {det_val_args}

In [ ]:
!python3 PaddleOCR/tools/eval.py -c PaddleOCR/configs/rec/PP-OCRv6/PP-OCRv6_small_rec.yml -o {rec_val_args}

### 5.4. Xuất Mô hình Inference (Export)
Mô hình sau huấn luyện cần được chuyển đổi sang định dạng **Inference** (sử dụng `tools/export_model.py`) để loại bỏ các tham số thừa của quá trình training, giúp tăng tốc độ dự đoán khi triển khai thực tế.

In [ ]:
!python3 PaddleOCR/tools/export_model.py -c PaddleOCR/configs/det/PP-OCRv6/PP-OCRv6_small_det.yml -o {det_export_args}

In [ ]:
!python3 PaddleOCR/tools/export_model.py -c PaddleOCR/configs/rec/PP-OCRv6/PP-OCRv6_small_rec.yml -o {rec_export_args}

In [ ]:
import requests
import os
import sys
from PIL import Image
import matplotlib.pyplot as plt
from paddleocr import PaddleOCR

urls = [
    "https://i.pinimg.com/736x/0d/af/b1/0dafb13a4db478a137573cace6b8c7f7.jpg",
    "https://i.pinimg.com/736x/ca/ec/ef/caecefb05e537e99108d2a04a8c46d3f.jpg",
    "https://i.pinimg.com/736x/ed/2f/73/ed2f738841d05fb2cdf35386a15143cb.jpg"
]

os.makedirs('test_images', exist_ok=True)
os.makedirs('output', exist_ok=True)
image_paths = []

for i, url in enumerate(urls):
    img_path = f'test_images/test_{i}.jpg'
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            with open(img_path, 'wb') as f:
                f.write(response.content)
            image_paths.append(img_path)
    except Exception as e:
        print(f"Lỗi khi tải {url}: {e}")

ocr = PaddleOCR(
    text_detection_model_name='PP-OCRv6_small_det',
    text_detection_model_dir='output/PP-OCRv6_small_det_infer/',
    text_recognition_model_name='PP-OCRv6_small_rec',
    text_recognition_model_dir='output/PP-OCRv6_small_rec_infer/',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    engine="paddle",
)

for img_path in image_paths:
    results = ocr.predict(img_path)

    for res in results:
        res.save_to_img("output")
        res.save_to_json("output")

output_images = [f for f in os.listdir('output') if f.endswith(('.jpg', '.png')) and 'res' in f.lower()]
if not output_images:
    output_images = [f for f in os.listdir('output') if f.endswith(('.jpg', '.png'))]

for out_img in sorted(output_images):
    path = os.path.join('output', out_img)
    img = Image.open(path)
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 6. Chuyển đổi sang ONNX
Chuyển đổi mô hình sang định dạng ONNX để tối ưu tốc độ và hiệu năng khi chạy trên edge device.

In [ ]:
!mkdir -p {det_onnx_model_dir}
!paddlex \
    --paddle2onnx \
    --paddle_model_dir {det_infer_model_dir} \
    --onnx_model_dir {det_onnx_model_dir} \
    --opset_version 11


In [ ]:
!mkdir -p {rec_onnx_model_dir}
!paddlex \
    --paddle2onnx \
    --paddle_model_dir {rec_infer_model_dir} \
    --onnx_model_dir {rec_onnx_model_dir} \
    --opset_version 11

## 7. Tải xuống kết quả huấn luyện

Tiến hành chạy lệnh nén các tệp đầu ra và tải về file `output.tar.gz`.


In [ ]:
!tar -czvf output.tar.gz output/